# Uber SQL Database - Unit 4 Practice Set 02 (Solutions)

This notebook contains 10 new WJEC-style Unit 4 SQL exercises using the Uber rideshare SQLite database.
All solution cells start with `%%sql` and are validated against `datasets/sqlite/uber_rideshare.db`.

Outputs are designed to stay small without depending on `LIMIT` in most tasks.

In [1]:
from pathlib import Path
import sys

notebooks_root = Path.cwd()
while not (notebooks_root / "helpers").exists() and notebooks_root != notebooks_root.parent:
    notebooks_root = notebooks_root.parent

sys.path.insert(0, str(notebooks_root))

from helpers import setup_uber_sql_notebook

setup_uber_sql_notebook()

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.
✓ Setup complete! Database ready.


PosixPath('/workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db')

## Important: run the setup code cell first

Run the setup code cell above before executing any SQL exercise.
It loads SQL magic and connects the notebook to the Uber SQLite database.

## Exercise 1 - High-demand trip mix (DQL: SELECT, WHERE, AND/OR, GROUP BY)

For trips requested on or after `2023-01-01`, count trips by `status` where either:
- `distance_km >= 45`, or
- `surge_multiplier >= 2.8`

Return `status` and `trip_count` ordered by highest count.

In [2]:
%%sql
SELECT status, COUNT(*) AS trip_count
FROM trips
WHERE requested_at >= '2023-01-01'
  AND (distance_km >= 45 OR surge_multiplier >= 2.8)
GROUP BY status
ORDER BY trip_count DESC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


status,trip_count
completed,399
cancelled,92
in_progress,4


## Exercise 2 - Premium-rated active drivers (DQL: SELECT, WHERE, IN)

List active drivers with very high ratings from model years 2012 or 2013.
Conditions:
- `is_active = 1`
- `vehicle_year IN (2012, 2013)`
- `rating >= 4.9`

Return `driver_id`, `vehicle_make`, `vehicle_model`, `rating`, `vehicle_year` in year then rating order.

In [3]:
%%sql
SELECT driver_id, vehicle_make, vehicle_model, rating, vehicle_year
FROM drivers
WHERE is_active = 1
  AND vehicle_year IN (2012, 2013)
  AND rating >= 4.9
ORDER BY vehicle_year ASC, rating DESC, driver_id ASC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


driver_id,vehicle_make,vehicle_model,rating,vehicle_year
98,Subaru,Outback,4.96,2012
178,Mazda,MX-5,5.0,2013
9,Nissan,Maxima,4.98,2013
76,Ford,Explorer,4.96,2013
127,Chevrolet,Silverado,4.91,2013
77,Ford,Explorer,4.9,2013


## Exercise 3 - Driver operational cancellations (DQL: IN, GROUP BY)

Count cancellations by `cancelled_by` for these operational reasons:
- `wrong pickup`
- `vehicle issue`
- `system error`

Return `cancelled_by` and `cancel_count`, highest count first.

In [4]:
%%sql
SELECT cancelled_by, COUNT(*) AS cancel_count
FROM cancellations
WHERE reason IN ('wrong pickup', 'vehicle issue', 'system error')
GROUP BY cancelled_by
ORDER BY cancel_count DESC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


cancelled_by,cancel_count
driver,398


## Exercise 4 - Completed fare by pickup city (Relational: JOIN + GROUP BY)

For completed trips, calculate by pickup city:
- number of trips (`completed_trips`)
- average fare (`avg_completed_fare` rounded to 2 dp)

Return `city`, `completed_trips`, `avg_completed_fare` ordered by highest average fare.

In [5]:
%%sql
SELECT l.city,
       COUNT(*) AS completed_trips,
       ROUND(AVG(t.total_fare), 2) AS avg_completed_fare
FROM trips AS t
JOIN locations AS l ON t.pickup_location_id = l.location_id
WHERE t.status = 'completed'
GROUP BY l.city
ORDER BY avg_completed_fare DESC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


city,completed_trips,avg_completed_fare
Los Angeles,3580,44.25
Houston,4432,42.42
New York,4836,29.29
Chicago,3979,29.28


## Exercise 5 - Above-average refund methods (Relational: subquery + GROUP BY)

Show payment methods whose refunded payment count is above the average refunded count per method.
Return `method` and `refunded_count` in descending count order.

In [6]:
%%sql
SELECT method, COUNT(*) AS refunded_count
FROM payments
WHERE status = 'refunded'
GROUP BY method
HAVING COUNT(*) > (
    SELECT AVG(refund_count)
    FROM (
        SELECT COUNT(*) AS refund_count
        FROM payments
        WHERE status = 'refunded'
        GROUP BY method
    ) AS per_method_refunds
)
ORDER BY refunded_count DESC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


method,refunded_count
wallet,54
card,51


## Exercise 6 - High-value refunded payments linked to trips (Relational: JOIN)

List refunded payments with amount at least 110 and include related trip parties.
Return `payment_id`, `trip_id`, `amount`, `rider_id`, `driver_id` ordered by amount descending.

In [7]:
%%sql
SELECT p.payment_id,
       p.trip_id,
       p.amount,
       t.rider_id,
       t.driver_id
FROM payments AS p
JOIN trips AS t ON p.trip_id = t.trip_id
WHERE p.status = 'refunded'
  AND p.amount >= 110
ORDER BY p.amount DESC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


payment_id,trip_id,amount,rider_id,driver_id
879,1016,145.94,91,37
12886,15331,137.44,1243,51
7230,8613,123.16,689,345
174,201,112.43,21,282


## Exercise 7 - Same-city completed flows (Relational: JOIN + GROUP BY)

Count completed trips where pickup city equals dropoff city.
Return `city` and `same_city_completed_trips` ordered by highest trip count.

In [8]:
%%sql
SELECT p.city, COUNT(*) AS same_city_completed_trips
FROM trips AS t
JOIN locations AS p ON t.pickup_location_id = p.location_id
JOIN locations AS d ON t.dropoff_location_id = d.location_id
WHERE t.status = 'completed'
  AND p.city = d.city
GROUP BY p.city
ORDER BY same_city_completed_trips DESC;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


city,same_city_completed_trips
New York,4836
Houston,4432
Chicago,3979
Los Angeles,3580


## Exercise 8 - Create dispatch flags table (DDL: CREATE TABLE)

Create a new table named `practice_dispatch_flags_02` with:
- `flag_id` as `PRIMARY KEY`
- `trip_id` as `NOT NULL`
- `dispatch_flag` as `NOT NULL`
- `priority_level` as `NOT NULL`
- `resolved` as `NOT NULL`

In [9]:
%%sql
CREATE TABLE IF NOT EXISTS practice_dispatch_flags_02 (
    flag_id INTEGER PRIMARY KEY,
    trip_id INTEGER NOT NULL,
    dispatch_flag TEXT NOT NULL,
    priority_level INTEGER NOT NULL,
    resolved INTEGER NOT NULL
);

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
Done.


[]

## Exercise 9 - Insert dispatch flags (DML: INSERT)

Insert the following rows into `practice_dispatch_flags_02`:
1. `(1, 102, 'surge-monitor', 2, 0)`
2. `(2, 408, 'driver-late', 3, 0)`
3. `(3, 915, 'rider-followup', 1, 1)`

In [10]:
%%sql
INSERT OR REPLACE INTO practice_dispatch_flags_02 (flag_id, trip_id, dispatch_flag, priority_level, resolved) VALUES
    (1, 102, 'surge-monitor', 2, 0),
    (2, 408, 'driver-late', 3, 0),
    (3, 915, 'rider-followup', 1, 1);

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
3 rows affected.


[]

## Exercise 10 - Resolve a dispatch flag (DML: UPDATE)

Set `resolved = 1` for `dispatch_flag = 'driver-late'`.
Then show all rows from `practice_dispatch_flags_02` ordered by `flag_id` to verify.

In [11]:
%%sql
UPDATE practice_dispatch_flags_02
SET resolved = 1
WHERE dispatch_flag = 'driver-late';

SELECT flag_id, trip_id, dispatch_flag, priority_level, resolved
FROM practice_dispatch_flags_02
ORDER BY flag_id;

 * sqlite:////workspaces/aButtTonOfSqlQuestions/datasets/sqlite/uber_rideshare.db
1 rows affected.
Done.


flag_id,trip_id,dispatch_flag,priority_level,resolved
1,102,surge-monitor,2,0
2,408,driver-late,3,1
3,915,rider-followup,1,1
